# ROBERTA SENTIMENT

In [ ]:
!pip install -q transformers torch tqdm

In [ ]:
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from transformers import pipeline

In [ ]:
df = pd.read_csv("Model Rule Based.csv")
df

In [ ]:
df.columns.tolist()

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print("Device:", device)

## Load Model RoBerta

In [ ]:
model = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

print("Model berhasil di-load")

In [ ]:
model("kamera bagus banget")

## Prediksi Batch

In [ ]:
from tqdm import tqdm
import torch

device = 0 if torch.cuda.is_available() else -1

komentar = (
    df["Comment_Final"]
    .astype(str)
    .fillna("")
    .tolist()
)

In [ ]:
hasil = []

batch_size = 32

for i in tqdm(range(0, len(komentar), batch_size)):

    batch = komentar[i:i+batch_size]

    pred = model(
        batch,
        truncation=True,
        max_length=512
    )

    hasil.extend(
        [x["label"] for x in pred]
    )

df["Sentiment_Roberta"] = hasil

print("✅ Prediksi selesai")

In [ ]:
df[
    ["Comment_Final","Sentiment_Roberta"]
].sample(10)

In [ ]:
df["Sentiment_Roberta"].value_counts()

## Ubah label Inggris → Indonesia

In [ ]:
map_label = {
    "positive": "Positif",
    "negative": "Negatif",
    "neutral": "Netral"
}

df["Sentiment_Roberta"] = (
    df["Sentiment_Roberta"]
    .replace(map_label)
)

In [ ]:
df[["Comment_Final","Sentiment_Rule","Sentiment_Roberta"]].sample(10)

## Crosstab Rule vs RoBERTa

In [ ]:
ct = pd.crosstab(
    df["Sentiment_Rule"],
    df["Sentiment_Roberta"]
)

display(ct)

In [ ]:
ct_percent = pd.crosstab(
    df["Sentiment_Rule"],
    df["Sentiment_Roberta"],
    normalize="index"
) * 100

ct_final = (
    ct.astype(str)
    + " ("
    + ct_percent.round(1).astype(str)
    + "%)"
)

display(ct_final)

## Distribusi Sentimen

In [ ]:
print("=== RULE BASED ===")
display(df["Sentiment_Rule"].value_counts())

In [ ]:
print("=== ROBERTA ===")
display(df["Sentiment_Roberta"].value_counts())

In [ ]:
df.to_csv(
    "Youtube Sentiment_Roberta.csv",
    index=False,
    encoding="utf-8-sig")

print("Youtube Sentiment_Roberta.csv tersimpan")